In [1]:
import numpy as np
atom_list = ['C', 'H', 'H', 'H', 'N', 'O']
Target_list = [0, 2, 3]
for target in Target_list:
    geom = np.load(str(target)+'/geom.npy')
    lines = []
    for iter in range(len(geom)):
        lines.append(str(len(atom_list))+'\n')
        lines.append(f'# Frame : {iter+1}\n')
        for i in range(len(atom_list)):
            atom = atom_list[i]
            lines.append(atom+' '+' '.join(f"{0.529177*x:.6f}" for x in geom[iter][i])+'\n')
    with open(str(target)+'/geom.xyz', 'w') as f:
        f.writelines(lines)

In [3]:
np.load('0/geom.npy')

array([[[ 2.08593828e+00, -2.49778330e-01,  0.00000000e+00],
        [ 3.55369234e+00,  1.19791251e+00, -3.59047964e-05],
        [ 2.22880346e+00, -1.44683668e+00, -1.68226443e+00],
        [ 2.22882992e+00, -1.44677243e+00,  1.68230790e+00],
        [-4.52952234e-01,  1.01936363e+00,  0.00000000e+00],
        [-2.16953598e+00, -4.92645931e-01,  0.00000000e+00]],

       [[ 2.09346962e+00, -2.55280490e-01,  7.54806133e-07],
        [ 3.54490458e+00,  1.20013813e+00, -3.84311551e-05],
        [ 2.24222955e+00, -1.44307161e+00, -1.67942406e+00],
        [ 2.24226198e+00, -1.44300521e+00,  1.67946993e+00],
        [-4.55910320e-01,  9.81159963e-01,  6.79386897e-06],
        [-2.19217962e+00, -4.58698020e-01, -7.43304913e-06]],

       [[ 2.08766417e+00, -2.55448628e-01,  1.72273896e-06],
        [ 3.53940075e+00,  1.19648028e+00, -4.16543912e-05],
        [ 2.23309062e+00, -1.44251249e+00, -1.67857120e+00],
        [ 2.23313227e+00, -1.44244346e+00,  1.67862017e+00],
        [-4.51743545

In [24]:
import numpy as np
from pyscf import gto, sftda
from pyscf.sftda.uhf_sf import get_ab_sf
from pyscf.geomopt import berny_solver, as_pyscf_method
from pyscf.grad import tduks_sf # this import is necessary.
try:
    import mcfun # mcfun>=0.2.5 must be used.
except ImportError:
    mcfun = None
from pyscf.data.nist import BOHR
import os
from pyscf.sftda.tools_td import transition_analyze

basis = 'cc-pvtz'
fun = 'B3LYP'

atom_list = ['C', 'H', 'H', 'H', 'N', 'O']
target_list = [0, 1, 2, 3]
e_list = []
i = 7
geom = np.load('3/geom.npy')[i]

temp = []
lines = []
for i in range(len(atom_list)):
    atom = atom_list[i]
    lines.append(atom+' '+' '.join(f"{x:.8f}" for x in geom[i]))
mol = gto.M(atom=lines, charge=0, spin=2, basis=basis, unit='Bohr')

mf = mol.UKS(xc=fun)
mf.kernel()
e1 = mf.e_tot

a, b = get_ab_sf(mf)
A_baba, A_abab = a
B_baab, B_abba = b

n_occ_a, n_virt_b = A_abab.shape[0], A_abab.shape[1]
n_occ_b, n_virt_a = B_abba.shape[2], B_abba.shape[3]

A_abab_2d = A_abab.reshape((n_occ_a*n_virt_b,n_occ_a*n_virt_b), order='C')
B_abba_2d = B_abba.reshape((n_occ_a*n_virt_b,n_occ_b*n_virt_a), order='C')
B_baab_2d = B_baab.reshape((n_occ_b*n_virt_a,n_occ_a*n_virt_b), order='C')
A_baba_2d = A_baba.reshape((n_occ_b*n_virt_a,n_occ_b*n_virt_a), order='C')

Casida_matrix = np.block([[ A_abab_2d, B_abba_2d],
                        [-B_baab_2d,-A_baba_2d]])

eigenvals, eigenvecs = np.linalg.eig(Casida_matrix)

idxt = eigenvals.real.argsort()
eigenvals = eigenvals[idxt]
eigenvecs = eigenvecs[:, idxt]

norms = np.linalg.norm(eigenvecs[:n_occ_a * n_virt_b], axis=0)**2
norms -= np.linalg.norm(eigenvecs[n_occ_a * n_virt_b:], axis=0)**2

sfd_eigenvals = eigenvals[norms>0]
sfd_eigenvecs = eigenvecs[:, norms>0].transpose(1,0)

def norm_xy(z):
    x = z[:n_occ_a*n_virt_b].reshape(n_occ_a,n_virt_b)
    y = z[n_occ_a*n_virt_b:].reshape(n_occ_b,n_virt_a)
    norm = np.linalg.norm(x)**2 - np.linalg.norm(y)**2
    norm = np.sqrt(1./norm)
    return ((0,x*norm), (y*norm,0))

mftd1 = sftda.TDA_SF(mf)
mftd1.nstates = len(sfd_eigenvals)
mftd1.extype = 1
mftd1.collinear_samples = 200
mftd1.e = sfd_eigenvals
mftd1.xy = [norm_xy(z) for z in sfd_eigenvecs]

for target in target_list:
    print(transition_analyze(mf, mftd1, mftd1.e[target], mftd1.xy[target], tdtype='TDDFT'))
    print(e1 + mftd1.e[target])
    temp.append(e1 + mftd1.e[target])

converged SCF energy = -169.818594172742  <S^2> = 2.0090112  2S+1 = 3.0060015
200
Excited energy : -5.946169855961909e-07 eV.
<S2> value : 2.048188150312145.
The main and second norm:
0.4861012447708371 0.48499730347017167
The main and second norm to orbital pair:
(13.0, 2.0) (12.0, 1.0)


None
-169.81859419459343
Excited energy : 0.556721237479496 eV.
<S2> value : 0.009574767746965307.
The main and second norm:
0.8062543401155741 0.1752278137248508
The main and second norm to orbital pair:
(13.0, 1.0) (12.0, 2.0)


None
-169.79813504282956
Excited energy : 0.7824610368153649 eV.
<S2> value : 0.009150029168712326.
The main and second norm:
0.489359692398811 0.488124097432062
The main and second norm to orbital pair:
(13.0, 2.0) (12.0, 1.0)


None
-169.7898392574885
Excited energy : 2.2049319892593577 eV.
<S2> value : 0.011776070323268772.
The main and second norm:
0.8106314416943061 0.17961305494277177
The main and second norm to orbital pair:
(12.0, 2.0) (13.0, 1.0)


None
-169.737564

In [2]:
temp

[-169.86507065571638,
 -169.83437718790591,
 -169.80889935687586,
 -169.6992575195343]

In [3]:
e_assign = {'S0': [], 'T1': [], 'S1': [], 'S2': []}

In [25]:
e_assign['T1'].append(temp[0])
e_assign['S0'].append(temp[1])
e_assign['S1'].append(temp[2])
e_assign['S2'].append(temp[3])

In [26]:
e_assign

{'S0': [-169.86507065571638,
  -169.84840384176363,
  -169.7744731121241,
  -169.81606176862377,
  -169.78947569622724,
  -169.7982163378476,
  -169.79884035237706,
  -169.79813504282956],
 'T1': [-169.83437718790591,
  -169.8356905619673,
  -169.79879357541603,
  -169.82532925973692,
  -169.8139681237233,
  -169.81882103557342,
  -169.81905594788597,
  -169.81859419459343],
 'S1': [-169.80889935687586,
  -169.8082536831976,
  -169.7698123336475,
  -169.7966232628199,
  -169.78514344150898,
  -169.78997271893164,
  -169.79025942778628,
  -169.7898392574885],
 'S2': [-169.6992575195343,
  -169.71944370774344,
  -169.71870456744222,
  -169.7311696639938,
  -169.73603196804936,
  -169.7371898541013,
  -169.73753252432903,
  -169.73756440958797]}

In [27]:
np.save('S2', np.array(e_assign))